In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0', 'python-dotenv>=1.0.0',
    'pyyaml>=6.0', 'requests>=2.32.0',
], check=True)

In [ ]:
import os, json, time, threading
from pathlib import Path
from datetime import datetime
import yaml, requests
from huggingface_hub import HfApi, CommitOperationAdd

WORK_DIR        = Path('/kaggle/working')
SEEDS_PATH      = WORK_DIR / 'seed_dialogues.jsonl'
AUGMENTED_PATH  = WORK_DIR / 'augmented_dialogues.jsonl'
DIALOGUES_PATH  = WORK_DIR / 'dialogues.jsonl'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p3c.json'
CONFIG_DIR      = Path('/kaggle/input/S2S-pipline-v2-0-2/config')

WAVE_SIZE_BYTES = 500 * 1024 * 1024
BATCH_SIZE      = 50
MAX_RETRIES     = 12
COMMIT_DELAY    = 3.0

In [ ]:
SECRETS = load_secrets(require_gemini=True)
HF_TOKEN    = SECRETS['HF_TOKEN_PRIMARY']
with open(CONFIG_DIR / 'hf_repos.yaml') as f: repos_cfg = yaml.safe_load(f)
STAGE2_REPO = repos_cfg['repos']['stage2_moe']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage2 repo: {STAGE2_REPO}')

In [ ]:
all_dialogues = []
for path in [SEEDS_PATH, AUGMENTED_PATH]:
    if path.exists():
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line: all_dialogues.append(json.loads(line))

print(f'[merge] seeds+augmented = {len(all_dialogues)} total dialogues')

domain_counts = {}
diff_counts   = {}
code_switch   = 0
for d in all_dialogues:
    domain_counts[d.get('domain','?')] = domain_counts.get(d.get('domain','?'), 0) + 1
    diff_counts[d.get('difficulty','?')] = diff_counts.get(d.get('difficulty','?'), 0) + 1
    if d.get('has_code_switch'): code_switch += 1

print(f'[merge] by domain: {domain_counts}')
print(f'[merge] by difficulty: {diff_counts}')
print(f'[merge] has_code_switch: {code_switch} ({100*code_switch/max(len(all_dialogues),1):.1f}%)')

with open(DIALOGUES_PATH, 'w', encoding='utf-8') as f:
    for d in all_dialogues:
        f.write(json.dumps(d, ensure_ascii=False) + '\n')

print(f'[merge] wrote {len(all_dialogues)} records to {DIALOGUES_PATH}')

In [ ]:
print('[upload] uploading dialogues.jsonl to stage2 repo...')
for attempt in range(MAX_RETRIES):
    try:
        HF_API.upload_file(
            path_or_fileobj=str(DIALOGUES_PATH),
            path_in_repo='dialogues.jsonl',
            repo_id=STAGE2_REPO,
            repo_type='dataset',
            commit_message=f'dialogues.jsonl — {len(all_dialogues)} dialogues',
        )
        print('[upload] dialogues.jsonl uploaded')
        break
    except Exception as e:
        wait = min(2**attempt, 120)
        print(f'  attempt {attempt+1}/{MAX_RETRIES} failed: {e} — retry in {wait}s')
        time.sleep(wait)

stats = {
    'total_dialogues': len(all_dialogues),
    'by_domain':       domain_counts,
    'by_difficulty':   diff_counts,
    'has_code_switch': code_switch,
    'generated_at':    datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
}
for attempt in range(MAX_RETRIES):
    try:
        HF_API.upload_file(
            path_or_fileobj=json.dumps(stats, indent=2).encode(),
            path_in_repo='stats.json',
            repo_id=STAGE2_REPO,
            repo_type='dataset',
            commit_message='stats.json',
        )
        print('[upload] stats.json uploaded')
        break
    except Exception as e:
        time.sleep(min(2**attempt, 120))

print(f'\n[done] stage2 repo: https://huggingface.co/datasets/{STAGE2_REPO}')
print('[done] pipeline 3 complete — ready for pipeline 4 (p4a_dummy_env.ipynb)')